# Final Analysis — Automated Pipeline

This notebook fills `finalAnalysis.csv` by:
1. Querying each RAG server (Vector, Graph, Hybrid) for answers
2. Collecting sources/triplets used
3. Computing RAGAS metrics (faithfulness, answer_relevancy, context_precision, context_recall)

**Prerequisites:**
- VectorRAG server running on `localhost:8000`
- GraphRAG server running on `localhost:8001`  
- HybridRAG server running on `localhost:8002`
- Document already uploaded (doc_id: `ceb7c087eb8c`)

**Resume-safe:** Every cell saves to CSV after each question and skips already-filled rows on re-run.
If an API key limit is hit mid-way, just change the key and re-run the same cell — it picks up where it left off.

In [4]:
# ============================================================
# Cell 0: Setup & Configuration
# ============================================================

import pandas as pd
import requests
import json
import time
import os
import sys

# --- Config ---
CSV_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'finalAnalysis.csv')
VECTOR_API  = 'http://localhost:8000'
GRAPH_API   = 'http://localhost:8001'
HYBRID_API  = 'http://localhost:8002'
DOCUMENT_ID = '4ec93fc12bff'

def is_empty(val):
    """Check if a cell value is empty/NaN/needs filling."""
    if pd.isna(val):
        return True
    s = str(val).strip()
    return s == '' or s == 'nan'

# Load the CSV
df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df)} questions from finalAnalysis.csv')
print(f'Columns: {list(df.columns)}')
print()
df[['question', 'question_type', 'ground_truth']].head(100)

Loaded 50 questions from finalAnalysis.csv
Columns: ['question', 'question_type', 'ground_truth', 'vectorRAG_answer', 'graphRAG_answer', 'hybridRAG_answer', 'vectorRAG_sources', 'graphRAG_triplets', 'hybridRAG_sources', 'vector_faithfulness', 'graph_faithfulness', 'hybrid_faithfulness', 'vector_answer_relevancy', 'graph_answer_relevancy', 'hybrid_answer_relevancy', 'vector_context_precision', 'graph_context_precision', 'hybrid_context_precision', 'vector_context_recall', 'graph_context_recall', 'hybrid_context_recall', 'combined_faithfulness', 'combined_answer_relevancy', 'combined_context_precision', 'combined_context_recall']



,question,question_type,ground_truth
0,What is the unique identification number (UIN)...,Basic Retrieval,The UIN of the policy is 105N153V04.
1,How does the policy document define 'Age' unde...,Basic Retrieval,'Age' means the age of the Life Assured in com...
2,What is the grace period allowed for a policyh...,Basic Retrieval,A grace period of 15 days from the premium due...
3,Where is the physical communication address of...,Basic Retrieval,"Customer Service Desk, ICICI Prudential Life I..."
4,After how many years from the date of policy i...,Basic Retrieval,No policy of Life Insurance shall be called in...
5,What are the three authorized frequencies for ...,Basic Retrieval,"The premiums can be paid in yearly, half-yearl..."
6,How much does the company charge for issuing a...,Basic Retrieval,The current charge for the issuance of a dupli...
7,Who is defined as an 'Appointee' under Part-B ...,Basic Retrieval,An Appointee is the person appointed by the po...
8,To which parties can benefits be legally payab...,Basic Retrieval,"Benefits are payable to the Policyholder, to t..."
9,What is the maximum compensation limit that th...,Basic Retrieval,The Ombudsman shall not award compensation exc...


In [7]:
# ============================================================
# Cell 1: Fill vectorRAG_answer & vectorRAG_sources
# ============================================================
# ► Skips rows that already have vectorRAG_answer filled.
# ► Saves CSV after EACH question so progress is never lost.

df = pd.read_csv(CSV_PATH)

print('=' * 60)
print('  Filling: vectorRAG_answer & vectorRAG_sources')
print('=' * 60)

skipped = 0
filled  = 0

for idx, row in df.iterrows():
    question = row['question']
    
    # --- SKIP if already filled ---
    if not is_empty(row.get('vectorRAG_answer')):
        skipped += 1
        print(f'[{idx+1}/{len(df)}] SKIP (already filled)')
        continue
    
    print(f'\n[{idx+1}/{len(df)}] {question[:80]}...')
    
    try:
        resp = requests.post(
            f'{VECTOR_API}/api/query',
            json={'document_id': DOCUMENT_ID, 'query': question, 'top_k': 5},
            timeout=120
        )
        resp.raise_for_status()
        data = resp.json()
        
        df.at[idx, 'vectorRAG_answer'] = data.get('answer', '')
        sources = data.get('sources', [])
        df.at[idx, 'vectorRAG_sources'] = json.dumps(sources)
        
        print(f'  ✓ Answer: {data["answer"][:100]}...')
        print(f'  ✓ Sources: {len(sources)} chunks retrieved')
        filled += 1
        
    except Exception as e:
        print(f'  ✗ ERROR: {e}')
        print(f'  ⚠ Stopping here. Change API key if needed and re-run this cell.')
        df.to_csv(CSV_PATH, index=False)
        raise  # Stop execution so user can fix the issue
    
    # --- SAVE after each question ---
    df.to_csv(CSV_PATH, index=False)
    time.sleep(1)  # Rate limiting

print(f'\n{"=" * 60}')
print(f'  ✓ Done! Skipped: {skipped}, Newly filled: {filled}')
print(f'  ✓ Saved to CSV')
print('=' * 60)

  Filling: vectorRAG_answer & vectorRAG_sources

[1/50] What is the unique identification number (UIN) of the ICICI Pru Future Perfect p...


C:\Users\grove\AppData\Local\Temp\ipykernel_32788\1652356306.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The unique identification number (UIN) of the ICICI Pru Future Perfect policy is 105N153V04. 
[Source: Passage 2, Page 1]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, 'vectorRAG_answer'] = data.get('answer', '')
C:\Users\grove\AppData\Local\Temp\ipykernel_32788\1652356306.py:38: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[{"text": "later  of  Policy  Issue  Date  or  Policy  Acceptance  Date. 6.Date  of\nDiscontinuance of the Policy means the due date of the first unpaid\npremium. 7.Date of Maturity means the date specified in the Policy\nSchedule on which the Policy Term ends and the Maturity benefit, if\napplicable, is payable. 8.Death Benefit means

  ✓ Answer: The unique identification number (UIN) of the ICICI Pru Future Perfect policy is 105N153V04. 
[Sourc...
  ✓ Sources: 5 chunks retrieved

[2/50] How does the policy document define 'Age' under Part-B?...
  ✓ Answer: According to Part-B of the policy document, 'Age' is defined as "age of the Life Assured in complete...
  ✓ Sources: 5 chunks retrieved

[3/50] What is the grace period allowed for a policyholder who has chosen a monthly pre...
  ✓ Answer: The grace period allowed for a policyholder who has chosen a monthly premium frequency is 15 days fo...
  ✓ Sources: 5 chunks retrieved

[4/50] Where is the physical communication address of the ICICI Prudential Life Insuran...
  ✓ Answer: The physical communication address of the ICICI Prudential Life Insurance Customer Service Desk is l...
  ✓ Sources: 5 chunks retrieved

[5/50] After how many years from the date of policy issuance, risk commencement, reviva...
  ✓ Answer: A life insurance policy cannot be questioned on any g

In [11]:
# ============================================================
# Cell 2: Fill graphRAG_answer & graphRAG_triplets
# ============================================================
# ► Skips rows that already have graphRAG_answer filled.
# ► Saves CSV after EACH question.

df = pd.read_csv(CSV_PATH)

print('=' * 60)
print('  Filling: graphRAG_answer & graphRAG_triplets')
print('=' * 60)

skipped = 0
filled  = 0

for idx, row in df.iterrows():
    question = row['question']
    
    # --- SKIP if already filled ---
    if not is_empty(row.get('graphRAG_answer')):
        skipped += 1
        print(f'[{idx+1}/{len(df)}] SKIP (already filled)')
        continue
    
    print(f'\n[{idx+1}/{len(df)}] {question[:80]}...')
    
    try:
        resp = requests.post(
            f'{GRAPH_API}/api/graph/query',
            json={'document_id': DOCUMENT_ID, 'query': question},
            timeout=120
        )
        resp.raise_for_status()
        data = resp.json()
        
        df.at[idx, 'graphRAG_answer'] = data.get('answer', '')
        triplets = data.get('triplets_used', [])
        df.at[idx, 'graphRAG_triplets'] = json.dumps(triplets)
        
        print(f'  ✓ Answer: {data["answer"][:100]}...')
        print(f'  ✓ Triplets: {len(triplets)} used')
        filled += 1
        
    except Exception as e:
        print(f'  ✗ ERROR: {e}')
        print(f'  ⚠ Stopping here. Change API key if needed and re-run this cell.')
        df.to_csv(CSV_PATH, index=False)
        raise
    
    # --- SAVE after each question ---
    df.to_csv(CSV_PATH, index=False)
    time.sleep(1)

print(f'\n{"=" * 60}')
print(f'  ✓ Done! Skipped: {skipped}, Newly filled: {filled}')
print(f'  ✓ Saved to CSV')
print('=' * 60)

  Filling: graphRAG_answer & graphRAG_triplets
[1/50] SKIP (already filled)
[2/50] SKIP (already filled)
[3/50] SKIP (already filled)
[4/50] SKIP (already filled)
[5/50] SKIP (already filled)
[6/50] SKIP (already filled)
[7/50] SKIP (already filled)
[8/50] SKIP (already filled)
[9/50] SKIP (already filled)
[10/50] SKIP (already filled)
[11/50] SKIP (already filled)
[12/50] SKIP (already filled)
[13/50] SKIP (already filled)
[14/50] SKIP (already filled)
[15/50] SKIP (already filled)
[16/50] SKIP (already filled)
[17/50] SKIP (already filled)
[18/50] SKIP (already filled)
[19/50] SKIP (already filled)
[20/50] SKIP (already filled)
[21/50] SKIP (already filled)
[22/50] SKIP (already filled)
[23/50] SKIP (already filled)
[24/50] SKIP (already filled)
[25/50] SKIP (already filled)
[26/50] SKIP (already filled)
[27/50] SKIP (already filled)
[28/50] SKIP (already filled)
[29/50] SKIP (already filled)

[30/50] What are the specific conditions, time limits, and late payment interest rate ca...

In [12]:
# ============================================================
# Cell 3: Fill hybridRAG_answer & hybridRAG_sources
# ============================================================
# ► Skips rows that already have hybridRAG_answer filled.
# ► Saves CSV after EACH question.
# ► Requires vectorRAG_answer & graphRAG_answer to be filled first.

df = pd.read_csv(CSV_PATH)

print('=' * 60)
print('  Filling: hybridRAG_answer & hybridRAG_sources')
print('=' * 60)

skipped = 0
filled  = 0

for idx, row in df.iterrows():
    question = row['question']
    
    # --- SKIP if already filled ---
    if not is_empty(row.get('hybridRAG_answer')):
        skipped += 1
        print(f'[{idx+1}/{len(df)}] SKIP (already filled)')
        continue
    
    vector_answer = str(row.get('vectorRAG_answer', '') or '')
    graph_answer  = str(row.get('graphRAG_answer', '') or '')
    
    print(f'\n[{idx+1}/{len(df)}] {question[:80]}...')
    
    try:
        resp = requests.post(
            f'{HYBRID_API}/api/hybrid/compose',
            json={
                'query': question,
                'vector_answer': vector_answer,
                'graph_answer': graph_answer,
            },
            timeout=120
        )
        resp.raise_for_status()
        data = resp.json()
        
        df.at[idx, 'hybridRAG_answer'] = data.get('answer', '')
        
        # Hybrid sources = combination of vector sources + graph triplets
        vector_sources_str = str(row.get('vectorRAG_sources', '[]') or '[]')
        graph_triplets_str = str(row.get('graphRAG_triplets', '[]') or '[]')
        
        try:
            v_sources = json.loads(vector_sources_str)
        except:
            v_sources = []
        try:
            g_triplets = json.loads(graph_triplets_str)
        except:
            g_triplets = []
        
        hybrid_sources = {
            'vector_sources': v_sources,
            'graph_triplets': g_triplets
        }
        df.at[idx, 'hybridRAG_sources'] = json.dumps(hybrid_sources)
        
        print(f'  ✓ Answer: {data["answer"][:100]}...')
        filled += 1
        
    except Exception as e:
        print(f'  ✗ ERROR: {e}')
        print(f'  ⚠ Stopping here. Change API key if needed and re-run this cell.')
        df.to_csv(CSV_PATH, index=False)
        raise
    
    # --- SAVE after each question ---
    df.to_csv(CSV_PATH, index=False)
    time.sleep(1)

print(f'\n{"=" * 60}')
print(f'  ✓ Done! Skipped: {skipped}, Newly filled: {filled}')
print(f'  ✓ Saved to CSV')
print('=' * 60)

  Filling: hybridRAG_answer & hybridRAG_sources

[1/50] What is the unique identification number (UIN) of the ICICI Pru Future Perfect p...


C:\Users\grove\AppData\Local\Temp\ipykernel_32788\1209779836.py:44: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The unique identification number (UIN) of the ICICI Pru Future Perfect policy is 105N153V04.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, 'hybridRAG_answer'] = data.get('answer', '')
C:\Users\grove\AppData\Local\Temp\ipykernel_32788\1209779836.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '{"vector_sources": [{"text": "later  of  Policy  Issue  Date  or  Policy  Acceptance  Date. 6.Date  of\nDiscontinuance of the Policy means the due date of the first unpaid\npremium. 7.Date of Maturity means the date specified in the Policy\nSchedule on which the Policy Term ends and the Maturity benefit, if\napplicable, is payable. 8.Death Benefit means the benef

  ✓ Answer: The unique identification number (UIN) of the ICICI Pru Future Perfect policy is 105N153V04....

[2/50] How does the policy document define 'Age' under Part-B?...
  ✓ Answer: According to the policy document, 'Age' under Part-B is defined as the age of the Life Assured in co...

[3/50] What is the grace period allowed for a policyholder who has chosen a monthly pre...
  ✓ Answer: The grace period allowed for a policyholder who has chosen a monthly premium frequency is 15 days fo...

[4/50] Where is the physical communication address of the ICICI Prudential Life Insuran...
  ✓ Answer: The physical communication address of the ICICI Prudential Life Insurance Customer Service Desk is l...

[5/50] After how many years from the date of policy issuance, risk commencement, reviva...
  ✓ Answer: A life insurance policy cannot be questioned on any ground whatsoever after 3 years from the date of...

[6/50] What are the three authorized frequencies for premium payments under this pol

In [13]:
# ============================================================
# Cell 4: Setup RAGAS evaluation (LLM + Embeddings)
# ============================================================
# This cell configures the LLM and EMBEDDINGS for RAGAS.
# ► answer_relevancy REQUIRES embeddings — without this it returns NaN.
# ► Re-run this cell after changing the API key in .env.

# !pip install ragas langchain-openai langchain-community datasets

import os
from dotenv import load_dotenv

# Load .env from project root (override=True to pick up new keys)
env_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '.env')
load_dotenv(dotenv_path=env_path, override=True)

OPENAI_API_KEY = os.getenv('OPEN_API_KEY') or os.getenv('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

print(f'OpenAI API Key loaded: {"Yes" if OPENAI_API_KEY else "No"}')
print(f'Key prefix: {OPENAI_API_KEY[:20]}...' if OPENAI_API_KEY else 'NOT SET')

# --- Setup LLM + Embeddings for RAGAS ---
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_llm = LangchainLLMWrapper(ChatOpenAI(
    model='gpt-4o',
    api_key=OPENAI_API_KEY,
    temperature=0,
))

ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(
    model='text-embedding-3-small',
    api_key=OPENAI_API_KEY,
))

print('\n✓ RAGAS LLM configured:        gpt-4o')
print('✓ RAGAS Embeddings configured:  text-embedding-3-small')
print('\nTip: If you change the API key in .env, re-run this cell to reload it.')

OpenAI API Key loaded: Yes
Key prefix: sk-proj-Jh16sKomVbMT...


c:\Users\grove\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\grove\miniconda3\Lib\site-packages\instructor\providers\gemini\client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]
C:\Users\grove\AppData\Local\Temp\ipykernel_32788\1965011540.py:28: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini


✓ RAGAS LLM configured:        gpt-4o
✓ RAGAS Embeddings configured:  text-embedding-3-small

Tip: If you change the API key in .env, re-run this cell to reload it.


C:\Users\grove\AppData\Local\Temp\ipykernel_32788\1965011540.py:34: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(


In [14]:
# ============================================================
# Cell 5: RAGAS helper functions
# ============================================================

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from datasets import Dataset


def batch_ragas_evaluate(questions, answers, contexts_list, ground_truths):
    """
    Batch-evaluate RAGAS metrics for multiple questions at once.
    This is MUCH faster than calling evaluate() per question
    because it parallelizes LLM calls internally.
    
    Returns:
        List of dicts, one per question, with keys:
        faithfulness, answer_relevancy, context_precision, context_recall
    """
    # Sanitize inputs
    clean_contexts = []
    for ctx in contexts_list:
        if not ctx:
            clean_contexts.append(['No context retrieved.'])
        else:
            clean_contexts.append(ctx)
    
    data = {
        'question': questions,
        'answer': answers,
        'contexts': clean_contexts,
        'ground_truth': ground_truths,
    }
    
    dataset = Dataset.from_dict(data)
    
    result = evaluate(
        dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    )
    
    # Convert result to per-row dicts
    result_df = result.to_pandas()
    per_row = []
    for _, row in result_df.iterrows():
        per_row.append({
            'faithfulness': row.get('faithfulness'),
            'answer_relevancy': row.get('answer_relevancy'),
            'context_precision': row.get('context_precision'),
            'context_recall': row.get('context_recall'),
        })
    return per_row


def extract_context_texts_from_sources(sources_json_str: str) -> list:
    """Extract plain text contexts from vectorRAG sources JSON."""
    try:
        sources = json.loads(str(sources_json_str))
        if isinstance(sources, list):
            return [s.get('text', '') for s in sources if s.get('text')]
    except:
        pass
    return []


def extract_context_texts_from_triplets(triplets_json_str: str) -> list:
    """Convert graphRAG triplets to plain text contexts."""
    try:
        triplets = json.loads(str(triplets_json_str))
        if isinstance(triplets, list):
            texts = []
            for t in triplets:
                subj = t.get('subject', '?')
                pred = t.get('predicate', '?').replace('_', ' ').lower()
                obj  = t.get('object', '?')
                texts.append(f'{subj} {pred} {obj}')
            return texts if texts else []
    except:
        pass
    return []


def extract_context_texts_from_hybrid(hybrid_json_str: str) -> list:
    """Extract combined contexts from hybridRAG sources JSON."""
    try:
        hybrid = json.loads(str(hybrid_json_str))
        contexts = []
        for s in hybrid.get('vector_sources', []):
            if s.get('text'):
                contexts.append(s['text'])
        for t in hybrid.get('graph_triplets', []):
            subj = t.get('subject', '?')
            pred = t.get('predicate', '?').replace('_', ' ').lower()
            obj  = t.get('object', '?')
            contexts.append(f'{subj} {pred} {obj}')
        return contexts if contexts else []
    except:
        pass
    return []


print('✓ RAGAS helper functions defined (batch mode).')

✓ RAGAS helper functions defined (batch mode).


C:\Users\grove\AppData\Local\Temp\ipykernel_32788\3361628844.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\grove\AppData\Local\Temp\ipykernel_32788\3361628844.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\grove\AppData\Local\Temp\ipykernel_32788\3361628844.py:6: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\grove\AppData\Local\Temp\ipykernel_32

In [15]:
# ============================================================
# Cell 6: Fill VectorRAG RAGAS metrics (BATCH)
# ============================================================
# Columns: vector_faithfulness, vector_answer_relevancy,
#          vector_context_precision, vector_context_recall
#
# ► Collects all unfilled rows → evaluates in ONE batch call.
# ► Much faster than per-question evaluation.
# ► Saves results to CSV after the batch completes.

df = pd.read_csv(CSV_PATH)

print('=' * 60)
print('  Computing RAGAS metrics for VectorRAG (BATCH)')
print('=' * 60)

# Find unfilled rows
unfilled_indices = []
for idx, row in df.iterrows():
    if is_empty(row.get('vector_faithfulness')):
        unfilled_indices.append(idx)
    else:
        print(f'[{idx+1}/{len(df)}] SKIP (already computed)')

if not unfilled_indices:
    print('\n✓ All rows already computed. Nothing to do.')
else:
    print(f'\n→ {len(unfilled_indices)} questions to evaluate (skipped {len(df) - len(unfilled_indices)})')
    
    # Prepare batch data
    questions = []
    answers = []
    contexts_list = []
    ground_truths = []
    
    for idx in unfilled_indices:
        row = df.iloc[idx]
        questions.append(row['question'])
        answers.append(str(row.get('vectorRAG_answer', '') or ''))
        ground_truths.append(str(row.get('ground_truth', '') or ''))
        contexts_list.append(extract_context_texts_from_sources(
            str(row.get('vectorRAG_sources', '[]') or '[]')
        ))
    
    print(f'→ Running batch RAGAS evaluation...')
    t0 = time.time()
    
    try:
        results = batch_ragas_evaluate(questions, answers, contexts_list, ground_truths)
        
        # Write results back
        for i, idx in enumerate(unfilled_indices):
            df.at[idx, 'vector_faithfulness']      = results[i]['faithfulness']
            df.at[idx, 'vector_answer_relevancy']  = results[i]['answer_relevancy']
            df.at[idx, 'vector_context_precision'] = results[i]['context_precision']
            df.at[idx, 'vector_context_recall']    = results[i]['context_recall']
        
        elapsed = time.time() - t0
        df.to_csv(CSV_PATH, index=False)
        print(f'\n✓ Batch complete! {len(unfilled_indices)} questions evaluated in {elapsed:.1f}s')
        print(f'✓ Saved to CSV')
        
    except Exception as e:
        print(f'\n✗ Batch evaluation failed: {e}')
        print(f'⚠ Change API key in .env → re-run Cell 4 → re-run this cell')
        raise

print('=' * 60)

  Computing RAGAS metrics for VectorRAG (BATCH)

→ 50 questions to evaluate (skipped 0)
→ Running batch RAGAS evaluation...


Evaluating:  47%|████▋     | 94/200 [08:06<11:57,  6.77s/it]Exception raised in Job[79]: TimeoutError()
Exception raised in Job[78]: TimeoutError()
Exception raised in Job[83]: TimeoutError()
Evaluating:  55%|█████▌    | 110/200 [09:54<17:35, 11.72s/it]Exception raised in Job[94]: TimeoutError()
Exception raised in Job[98]: TimeoutError()
Exception raised in Job[106]: APIConnectionError(Connection error.)
Exception raised in Job[118]: APIConnectionError(Connection error.)
Exception raised in Job[110]: APIConnectionError(Connection error.)
Exception raised in Job[117]: APIConnectionError(Connection error.)
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[129]: RateLimitError(Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quo


✓ Batch complete! 50 questions evaluated in 1162.6s
✓ Saved to CSV


In [16]:
# ============================================================
# Cell 7: Fill GraphRAG RAGAS metrics (BATCH)
# ============================================================
# Columns: graph_faithfulness, graph_answer_relevancy,
#          graph_context_precision, graph_context_recall

df = pd.read_csv(CSV_PATH)

print('=' * 60)
print('  Computing RAGAS metrics for GraphRAG (BATCH)')
print('=' * 60)

# Find unfilled rows
unfilled_indices = []
for idx, row in df.iterrows():
    if is_empty(row.get('graph_faithfulness')):
        unfilled_indices.append(idx)
    else:
        print(f'[{idx+1}/{len(df)}] SKIP (already computed)')

if not unfilled_indices:
    print('\n✓ All rows already computed. Nothing to do.')
else:
    print(f'\n→ {len(unfilled_indices)} questions to evaluate (skipped {len(df) - len(unfilled_indices)})')
    
    questions = []
    answers = []
    contexts_list = []
    ground_truths = []
    
    for idx in unfilled_indices:
        row = df.iloc[idx]
        questions.append(row['question'])
        answers.append(str(row.get('graphRAG_answer', '') or ''))
        ground_truths.append(str(row.get('ground_truth', '') or ''))
        contexts_list.append(extract_context_texts_from_triplets(
            str(row.get('graphRAG_triplets', '[]') or '[]')
        ))
    
    print(f'→ Running batch RAGAS evaluation...')
    t0 = time.time()
    
    try:
        results = batch_ragas_evaluate(questions, answers, contexts_list, ground_truths)
        
        for i, idx in enumerate(unfilled_indices):
            df.at[idx, 'graph_faithfulness']      = results[i]['faithfulness']
            df.at[idx, 'graph_answer_relevancy']  = results[i]['answer_relevancy']
            df.at[idx, 'graph_context_precision'] = results[i]['context_precision']
            df.at[idx, 'graph_context_recall']    = results[i]['context_recall']
        
        elapsed = time.time() - t0
        df.to_csv(CSV_PATH, index=False)
        print(f'\n✓ Batch complete! {len(unfilled_indices)} questions evaluated in {elapsed:.1f}s')
        print(f'✓ Saved to CSV')
        
    except Exception as e:
        print(f'\n✗ Batch evaluation failed: {e}')
        print(f'⚠ Change API key in .env → re-run Cell 4 → re-run this cell')
        raise

print('=' * 60)

  Computing RAGAS metrics for GraphRAG (BATCH)

→ 50 questions to evaluate (skipped 0)
→ Running batch RAGAS evaluation...


Evaluating:   7%|▋         | 14/200 [03:00<18:02,  5.82s/it]Exception raised in Job[11]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[16]: RateLimitError(Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}})
Evaluating:  18%|█▊        | 36/200 [06:39<28:29, 10.42s/it]Exception raised in Job[36]: RateLimitError(Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}})
Exception raised in Job[39]: RateLimitError(Error code: 429 - {'error': {'message


✓ Batch complete! 50 questions evaluated in 1893.8s
✓ Saved to CSV


In [17]:
# ============================================================
# Cell 8: Fill HybridRAG RAGAS metrics (BATCH)
# ============================================================
# Columns: hybrid_faithfulness, hybrid_answer_relevancy,
#          hybrid_context_precision, hybrid_context_recall

df = pd.read_csv(CSV_PATH)

print('=' * 60)
print('  Computing RAGAS metrics for HybridRAG (BATCH)')
print('=' * 60)

# Find unfilled rows
unfilled_indices = []
for idx, row in df.iterrows():
    if is_empty(row.get('hybrid_faithfulness')):
        unfilled_indices.append(idx)
    else:
        print(f'[{idx+1}/{len(df)}] SKIP (already computed)')

if not unfilled_indices:
    print('\n✓ All rows already computed. Nothing to do.')
else:
    print(f'\n→ {len(unfilled_indices)} questions to evaluate (skipped {len(df) - len(unfilled_indices)})')
    
    questions = []
    answers = []
    contexts_list = []
    ground_truths = []
    
    for idx in unfilled_indices:
        row = df.iloc[idx]
        questions.append(row['question'])
        answers.append(str(row.get('hybridRAG_answer', '') or ''))
        ground_truths.append(str(row.get('ground_truth', '') or ''))
        contexts_list.append(extract_context_texts_from_hybrid(
            str(row.get('hybridRAG_sources', '{}') or '{}')
        ))
    
    print(f'→ Running batch RAGAS evaluation...')
    t0 = time.time()
    
    try:
        results = batch_ragas_evaluate(questions, answers, contexts_list, ground_truths)
        
        for i, idx in enumerate(unfilled_indices):
            df.at[idx, 'hybrid_faithfulness']      = results[i]['faithfulness']
            df.at[idx, 'hybrid_answer_relevancy']  = results[i]['answer_relevancy']
            df.at[idx, 'hybrid_context_precision'] = results[i]['context_precision']
            df.at[idx, 'hybrid_context_recall']    = results[i]['context_recall']
        
        elapsed = time.time() - t0
        df.to_csv(CSV_PATH, index=False)
        print(f'\n✓ Batch complete! {len(unfilled_indices)} questions evaluated in {elapsed:.1f}s')
        print(f'✓ Saved to CSV')
        
    except Exception as e:
        print(f'\n✗ Batch evaluation failed: {e}')
        print(f'⚠ Change API key in .env → re-run Cell 4 → re-run this cell')
        raise

print('=' * 60)

  Computing RAGAS metrics for HybridRAG (BATCH)

→ 50 questions to evaluate (skipped 0)
→ Running batch RAGAS evaluation...


Evaluating:  24%|██▍       | 49/200 [08:32<27:59, 11.13s/it]Exception raised in Job[47]: RateLimitError(Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}})
Exception raised in Job[54]: RateLimitError(Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}})
Evaluating:  30%|███       | 60/200 [10:05<20:28,  8.77s/it]Exception raised in Job[65]: RateLimitError(Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For mo


✓ Batch complete! 50 questions evaluated in 1835.1s
✓ Saved to CSV


In [18]:
# ============================================================
# Cell 9: Fill combined metrics (averages)
# ============================================================
# Columns: combined_faithfulness, combined_answer_relevancy,
#          combined_context_precision, combined_context_recall

import numpy as np

df = pd.read_csv(CSV_PATH)

print('=' * 60)
print('  Computing Combined (Average) Metrics')
print('=' * 60)

metric_names = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
rag_prefixes = ['vector', 'graph', 'hybrid']

skipped = 0
filled  = 0

for idx, row in df.iterrows():
    question = row['question']
    
    # --- SKIP if already computed ---
    if not is_empty(row.get('combined_faithfulness')):
        skipped += 1
        print(f'[{idx+1}/{len(df)}] SKIP (already computed)')
        continue
    
    print(f'[{idx+1}/{len(df)}] {question[:60]}...')
    
    for metric in metric_names:
        values = []
        for prefix in rag_prefixes:
            col = f'{prefix}_{metric}'
            val = row.get(col)
            if pd.notna(val):
                try:
                    values.append(float(val))
                except (ValueError, TypeError):
                    pass
        
        if values:
            df.at[idx, f'combined_{metric}'] = round(np.mean(values), 4)
        else:
            df.at[idx, f'combined_{metric}'] = None
    
    filled += 1

# Save
df.to_csv(CSV_PATH, index=False)
print(f'\n{"=" * 60}')
print(f'  ✓ Done! Skipped: {skipped}, Newly computed: {filled}')
print('=' * 60)

  Computing Combined (Average) Metrics
[1/50] What is the unique identification number (UIN) of the ICICI ...
[2/50] How does the policy document define 'Age' under Part-B?...
[3/50] What is the grace period allowed for a policyholder who has ...
[4/50] Where is the physical communication address of the ICICI Pru...
[5/50] After how many years from the date of policy issuance, risk ...
[6/50] What are the three authorized frequencies for premium paymen...
[7/50] How much does the company charge for issuing a duplicate cop...
[8/50] Who is defined as an 'Appointee' under Part-B of this policy...
[9/50] To which parties can benefits be legally payable according t...
[10/50] What is the maximum compensation limit that the Insurance Om...
[11/50] If a policyholder opts for a monthly premium payment mode, w...
[12/50] What is the Guaranteed Surrender Value (GSV) factor for a po...
[13/50] For a policy with a Premium Payment Term of 10 years, what i...
[14/50] What was the specific policy lo

In [19]:
# ============================================================
# Cell 10: Final Summary & Verification
# ============================================================

df = pd.read_csv(CSV_PATH)

print('=' * 60)
print('  FINAL ANALYSIS SUMMARY')
print('=' * 60)
print(f'\nTotal questions: {len(df)}')
print(f'Columns: {len(df.columns)}')
print()

# Check completeness
for col in df.columns:
    non_null = df[col].notna().sum()
    pct = (non_null / len(df)) * 100
    status = '✓' if pct == 100 else '⚠' if pct > 0 else '✗'
    print(f'  {status} {col}: {non_null}/{len(df)} filled ({pct:.0f}%)')

print()
print('─' * 60)
print('METRIC AVERAGES ACROSS ALL QUESTIONS')
print('─' * 60)

metric_cols = [
    'vector_faithfulness', 'vector_answer_relevancy', 'vector_context_precision', 'vector_context_recall',
    'graph_faithfulness', 'graph_answer_relevancy', 'graph_context_precision', 'graph_context_recall',
    'hybrid_faithfulness', 'hybrid_answer_relevancy', 'hybrid_context_precision', 'hybrid_context_recall',
    'combined_faithfulness', 'combined_answer_relevancy', 'combined_context_precision', 'combined_context_recall',
]

for col in metric_cols:
    if col in df.columns:
        mean_val = pd.to_numeric(df[col], errors='coerce').mean()
        print(f'  {col}: {mean_val:.4f}')

print()
print('✓ All done! finalAnalysis.csv is fully populated.')
df.head(20)

  FINAL ANALYSIS SUMMARY

Total questions: 50
Columns: 25

  ✓ question: 50/50 filled (100%)
  ✓ question_type: 50/50 filled (100%)
  ✓ ground_truth: 50/50 filled (100%)
  ✓ vectorRAG_answer: 50/50 filled (100%)
  ✓ graphRAG_answer: 50/50 filled (100%)
  ✓ hybridRAG_answer: 50/50 filled (100%)
  ✓ vectorRAG_sources: 50/50 filled (100%)
  ✓ graphRAG_triplets: 50/50 filled (100%)
  ✓ hybridRAG_sources: 50/50 filled (100%)
  ⚠ vector_faithfulness: 43/50 filled (86%)
  ✗ graph_faithfulness: 0/50 filled (0%)
  ✗ hybrid_faithfulness: 0/50 filled (0%)
  ⚠ vector_answer_relevancy: 28/50 filled (56%)
  ✗ graph_answer_relevancy: 0/50 filled (0%)
  ✗ hybrid_answer_relevancy: 0/50 filled (0%)
  ⚠ vector_context_precision: 23/50 filled (46%)
  ✗ graph_context_precision: 0/50 filled (0%)
  ✗ hybrid_context_precision: 0/50 filled (0%)
  ⚠ vector_context_recall: 45/50 filled (90%)
  ✗ graph_context_recall: 0/50 filled (0%)
  ✗ hybrid_context_recall: 0/50 filled (0%)
  ⚠ combined_faithfulness: 43/50 fi

,question,question_type,ground_truth,vectorRAG_answer,graphRAG_answer,hybridRAG_answer,vectorRAG_sources,graphRAG_triplets,hybridRAG_sources,vector_faithfulness,...,vector_context_precision,graph_context_precision,hybrid_context_precision,vector_context_recall,graph_context_recall,hybrid_context_recall,combined_faithfulness,combined_answer_relevancy,combined_context_precision,combined_context_recall
0,What is the unique identification number (UIN)...,Basic Retrieval,The UIN of the policy is 105N153V04.,The unique identification number (UIN) of the ...,⚠️ The requested information was not found in ...,The unique identification number (UIN) of the ...,"[{""text"": ""later of Policy Issue Date or ...","[{""subject"": ""ICICI Prudential Life Insurance ...","{""vector_sources"": [{""text"": ""later of Polic...",1.000000,...,0.500000,NaN,NaN,1.0,NaN,NaN,1.0000,1.0000,0.5000,1.0
1,How does the policy document define 'Age' unde...,Basic Retrieval,'Age' means the age of the Life Assured in com...,"According to Part-B of the policy document, 'A...",⚠️ The requested information was not found in ...,"According to the policy document, 'Age' under ...","[{""text"": ""Policy Document - Terms and Conditi...","[{""subject"": ""Policy document"", ""subject_type""...","{""vector_sources"": [{""text"": ""Policy Document ...",0.666667,...,1.000000,NaN,NaN,1.0,NaN,NaN,0.6667,0.8952,1.0000,1.0
2,What is the grace period allowed for a policyh...,Basic Retrieval,A grace period of 15 days from the premium due...,The grace period allowed for a policyholder wh...,To determine the grace period allowed for a po...,The grace period allowed for a policyholder wh...,"[{""text"": ""If you are unable to pay Instalment...","[{""subject"": ""Grace Period"", ""subject_type"": ""...","{""vector_sources"": [{""text"": ""If you are unabl...",0.250000,...,1.000000,NaN,NaN,1.0,NaN,NaN,0.2500,1.0000,1.0000,1.0
3,Where is the physical communication address of...,Basic Retrieval,"Customer Service Desk, ICICI Prudential Life I...",The physical communication address of the ICIC...,To find the physical communication address of ...,The physical communication address of the ICIC...,"[{""text"": ""Insurance Ombudsman. We request ...","[{""subject"": ""ICICI Prudential Life Insurance ...","{""vector_sources"": [{""text"": ""Insurance Ombud...",0.333333,...,0.500000,NaN,NaN,1.0,NaN,NaN,0.3333,1.0000,0.5000,1.0
4,After how many years from the date of policy i...,Basic Retrieval,No policy of Life Insurance shall be called in...,A life insurance policy cannot be questioned o...,"After analyzing the knowledge graph triplets, ...",A life insurance policy cannot be questioned o...,"[{""text"": ""from a) the date of issuance of pol...","[{""subject"": ""Policy of Life Insurance"", ""subj...","{""vector_sources"": [{""text"": ""from a) the date...",0.555556,...,NaN,NaN,NaN,1.0,NaN,NaN,0.5556,0.8846,NaN,1.0
5,What are the three authorized frequencies for ...,Basic Retrieval,"The premiums can be paid in yearly, half-yearl...",The three authorized frequencies for premium p...,The three authorized frequencies for premium p...,The three authorized frequencies for premium p...,"[{""text"": ""base premium and the extra mor...","[{""subject"": ""Premiums"", ""subject_type"": ""PAYM...","{""vector_sources"": [{""text"": ""base premium a...",1.000000,...,1.000000,NaN,NaN,1.0,NaN,NaN,1.0000,0.8493,1.0000,1.0
6,How much does the company charge for issuing a...,Basic Retrieval,The current charge for the issuance of a dupli...,The company charges ₹ 200 for issuing a duplic...,⚠️ The requested information was not found in ...,The company charges ₹ 200 for issuing a duplic...,"[{""text"": ""charges for issuance of duplicate p...","[{""subject"": ""duplicate policy document"", ""sub...","{""vector_sources"": [{""text"": ""charges for issu...",0.333333,...,1.000000,NaN,NaN,1.0,NaN,NaN,0.3333,1.0000,1.0000,1.0
7,Who is defined as an 'Appointee' under Part-B ...,Basic Retrieval,An Appointee is the person ap